In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/yvvswaraag@gmail.com/capgemini-data-engineering-training/Week3/IndividualProject_E2E_DE_Project_FMCG/1_Setup/utilities

In [0]:
# here these are imported from the utilities file so if we want to make any changes we can directly go to utilities and change and no need to change in this notebook
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","customers","Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(catalog,data_source)

In [0]:
# now we want to read a customer file which is in s3 bucket
base_path=f's3://swaraag-fcmg-indpro/{data_source}/*.csv'
print(base_path)

In [0]:
# now from this path we will create a dataframe
df = (
    spark.read.format('csv')
    .option("header",True)
    .option("inferSchema",True)
    .load(base_path)
    .withColumn("read_timestamp",f.current_timestamp())
    .select("*","_metadata.file_name","_metadata.file_size")
)
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
# now we will direclty write it to bronze layer
df.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
                .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

## Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

In [0]:
df_duplicates = df_bronze.groupBy("customer_id").count().filter(f.col("count")>1)
display(df_duplicates)

In [0]:
print(df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_id"])
print(df_silver.count())

In [0]:
display(
    df_silver.filter(f.col("customer_name")!=f.trim(f.col("customer_name")))
)

In [0]:
df_silver = df_silver.withColumn("customer_name",f.trim(f.col("customer_name")))
df_silver.display()

In [0]:
display(
    df_silver.filter(f.col("customer_name")!=f.trim(f.col("customer_name")))
)

In [0]:
df_silver.select('city').distinct().show()

In [0]:
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}


allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        f.when(f.col("city").isNull(), None)
         .when(f.col("city").isin(allowed), f.col("city"))
         .otherwise(None)
    )
)

In [0]:
# Sanity check
df_silver.select('city').distinct().show()

In [0]:
df_silver.select('customer_name').distinct().show()

In [0]:
# Title case fix
df_silver = df_silver.withColumn(
    "customer_name",
    f.when(f.col("customer_name").isNull(), None)
     .otherwise(f.initcap("customer_name"))
)

In [0]:
# sanity check

df_silver.select('customer_name').distinct().show()

In [0]:
df_silver.filter(f.col("city").isNull()).show(truncate=False)


In [0]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(f.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:

# Business Confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix)

In [0]:
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        "city",
        f.coalesce("city", "fixed_city")   # Replace null with fixed city
    )
    .drop("fixed_city")
)

In [0]:
# Sanity Checks

null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(f.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:
df_silver = df_silver.withColumn("customer_id", f.col("customer_id").cast("string"))
print(df_silver.printSchema())

In [0]:
df_silver = (
    df_silver
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        f.concat_ws("-", "customer_name", f.coalesce(f.col("city"), f.lit("Unknown")))
    )
    
    # Static attributes aligned with parent data model
    .withColumn("market", f.lit("India"))
    .withColumn("platform", f.lit("Sports Bar"))
    .withColumn("channel", f.lit("Acquisition"))
)

In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

## Gold processing

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")


# take req cols only
# "customer_id, customer_name, city, read_timestamp, file_name, file_size, customer, market, platform, channel"
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

## Merging Data source with parent

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    f.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()